In [1]:
import pandas as pd
from ortools.sat.python import cp_model
import random
from datetime import datetime, timedelta


In [2]:
people_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main People')
jobs_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main Desk')

# jobs_df
# people_df

In [3]:
# map all the value 
grade_map = {'A':1, 'B':2, 'C':3, 'D':4, 'F':5}
edu_map = {'Doc': 1, 'Degree':2, 'Uni':2, 'Dipolma':3, 'O Level':4, 'N Level':5}
health_map = {'Fit':1, 'Semi Fit':2, 'Not Fit': 3}
sec_map = {'CAT1':1, 'CAT2':2, 'CAT3':3, 'CAT4':4, 'CAT5':5, 'CAT6':6, 'CAT7':7, 'CAT8':8, 'CAT9':9, 'CAT10':10}

jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Desk_ID          942 non-null    object
 1   Req_Grade        942 non-null    object
 2   Req_Edu          942 non-null    object
 3   Req_Health       942 non-null    object
 4   Sec_Clerance     942 non-null    object
 5   Req_Appointment  942 non-null    object
dtypes: object(6)
memory usage: 44.3+ KB


In [4]:
people_df['Grade'] = people_df['Grade'].replace(grade_map)
people_df['Edu_Type'] = people_df['Edu_Type'].replace(edu_map)
people_df['Health'] = people_df['Health'].replace(health_map)
people_df['Security'] = people_df['Security'].replace(sec_map)

jobs_df['Req_Grade'] = jobs_df['Req_Grade'].replace(grade_map)
jobs_df['Req_Edu'] = jobs_df['Req_Edu'].replace(edu_map)
jobs_df['Req_Health'] = jobs_df['Req_Health'].replace(health_map)
jobs_df['Sec_Clerance'] = jobs_df['Sec_Clerance'].replace(sec_map)


C:\Users\Cherry\AppData\Local\Temp\ipykernel_8552\875482697.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  people_df['Grade'] = people_df['Grade'].replace(grade_map)
C:\Users\Cherry\AppData\Local\Temp\ipykernel_8552\875482697.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  people_df['Edu_Type'] = people_df['Edu_Type'].replace(edu_map)
C:\Users\Cherry\AppData\Local\Temp\ipykernel_8552\875482697.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old be

In [5]:
appointment_cols = ["Appointments_1", "Appointments_2", "Appointments_3", "Appointments _4"]

# Get all the appointment and put into 1 column
people_df["Appointments"] = (
    people_df[appointment_cols]
    .apply(lambda row: [a for a in row if pd.notna(a) and a != ""], axis=1)
)
people_df = people_df.drop(columns=appointment_cols)

people_df

,Name_ID,Name,Grade,Edu_Type,Health,Security,Appointments
0,P000001,Roxuqy Tiwoni,2,2,2,5,"[Business Development, Customer Support, Custo..."
1,P000002,Lyxu Subyzu,5,3,1,3,"[Logistics, Risk Management, Supply Chain, Ris..."
2,P000003,Cagyla Gedyzo,4,1,2,10,"[Corporate Affairs, Human Resources (HR), Inve..."
3,P000004,Huwyle Wola,3,2,3,7,"[Corporate Affairs, Internal Audit, Sales, Inv..."
4,P000005,Levyjo Buti,1,1,1,10,"[Facilities Management, Maintenance, Health & ..."
...,...,...,...,...,...,...,...
1992,P001993,Mozuni Ciqe,5,2,1,4,"[Quality Assurance (QA), Operations, Complianc..."
1993,P001994,Vogu Sysu,4,5,2,1,"[Project Management Office (PMO), Internal Aud..."
1994,P001995,Tybufe Cotysy,1,5,1,6,"[Health & Safety, Partnerships & Alliances, Go..."
1995,P001996,Viji Zuqemi,4,3,1,7,"[Corporate Affairs, Media Production, Public R..."


In [ ]:
# def str_time_prop(start, end, time_format, prop):
#     stime = time.mktime(time.strptime(start, time_format))
#     etime = time.mktime(time.strptime(end, time_format))
#     ptime = stime + prop * (etime - stime)
#     return time.strftime(time_format, time.localtime(ptime))

# def random_date(start, end, prop):
#     return str_time_prop(start, end, '%m/%d/%Y', prop)

# def start_date()
#     return random_date(f"1/1/{2017}", f"1/1/{2023}", random.random())

# def end_date(start_date)
#     start_year=start_date[-4:]
#     end_start, end_end = int(start_year) +2, int(start_year) +5
#     return random_date(f"1/1/{end_start}", f"1/1/{end_end}", random.random())
    
# print(start_date())

def str_time_prop(start, end, time_format, prop):
    stime = time.mktime(time.strptime(start, time_format))
    etime = time.mktime(time.strptime(end, time_format))
    ptime = stime + prop * (etime - stime)
    return time.strftime(time_format, time.localtime(ptime))

def random_date(start, end, prop):
    return str_time_prop(start, end, '%m/%d/%Y', prop)

def start_date():
    print(random_date("1/1/2017", "1/1/2023", random.random())
)
    # return random_date("1/1/2017", "1/1/2023", random.random())

# def end_date(start_date_str):
#     start_year = int(start_date_str[-4:])
#     end_start = start_year + 2
#     end_end = start_year + 5
#     return random_date(f"1/1/{end_start}", f"1/1/{end_end}", random.random())

# Example usage
s_date = start_date()
# e_date = end_date(s_date)

print("Start date:", s_date)
# print("End date:", e_date)

In [ ]:
people_df["Start Date"] = people_df.apply((start_date()), axis=1)
people_df

In [ ]:
print("Starting the model")

In [ ]:
model = cp_model.CpModel()
assign = {}

# Getting all people who matches the job base on hard constraints
for p_idx, person in people_df.iterrows():
    for j_idx, job in jobs_df.iterrows():
        if person['Health'] != job['Req_Health']:
            continue
        if person['Security'] != job['Sec_Clerance']:
            continue
        var = model.NewBoolVar(f"assign_p{p_idx}_j{j_idx}")
        assign[(p_idx, j_idx)] = var


In [ ]:
jobs_df.info()

In [ ]:
#Soft constraints
appointment_terms = []
edu_terms = []

for (p_idx, j_idx), var in assign.items():
    person_apps = people_df.loc[p_idx, "Appointments"]
    job_app = jobs_df.loc[j_idx, "Req_Appointment"]
    
    if job_app in person_apps:
        app_idx = person_apps.index(job_app)
        appoint_score = len(person_apps) - app_idx
    else:
        appoint_score = 0
    appointment_terms.append(var * appoint_score * 5)  # weight (higher the better)

    edu_score = 1 if people_df.loc[p_idx, "Edu_Type"] == jobs_df.loc[j_idx, "Req_Edu"] else 0
    edu_terms.append(var * edu_score * 1) 

In [ ]:
model.Maximize(sum(appointment_terms) + sum(edu_terms) )

solver = cp_model.CpSolver()
status = solver.Solve(model)

In [ ]:
results = []

for (p_idx, j_idx), var in assign.items():
    if solver.BooleanValue(var):
        appoint_match = (
            jobs_df.loc[j_idx, "Req_Appointment"]
            in people_df.loc[p_idx, "Appointments"]
        )
        edu_match = (
            people_df.loc[p_idx, "Edu_Type"]
            == jobs_df.loc[j_idx, "Req_Edu"]
        )

        results.append({
            "Person": people_df.loc[p_idx, "Name"],
            "Job": jobs_df.loc[j_idx, "Desk_ID"],
            "Appointment_Match": int(appoint_match),
            "Edu_Match": int(edu_match),
            "Suitable": int(appoint_match) + int(edu_match)
        })

result_df = pd.DataFrame(results)
print(result_df)

In [ ]:
suitable_df = result_df.loc[result_df['Suitable'] == 2]
suitable_df

In [ ]:
result_df.to_excel("Result.xlsx")